<a href="https://colab.research.google.com/github/LikithaC17/GramaSetu-AI/blob/main/GramaSetu_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("Running on CPU")

PyTorch version: 2.14.1+cu130
GPU available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [2]:
!pip install -U git+https://github.com/huggingface/transformers.git accelerate peft safetensors

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-ylxkf3nx
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-ylxkf3nx
  Resolved https://github.com/huggingface/transformers.git to commit 1c9278519d5e6fcc1d13044bed63e4ab3448be28
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.6/863.6 kB 31.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 886.7/886.7 kB 43.1 MB/s  0:00:00
  Created wheel for transformers: filename=transformers-5.25.0.dev0-py3-none-any.whl size=15047710 sha256=eb6a0fe98a6864656beb9171ff3b5f9d68259e2c484a247fd565787fb17d409b
  Stored in directory: /tmp/pip-ephem-wheel-cache-8ns3d6hs/wheels/90/86/bb/8ad35b428289429952d8316d30f64abedae0ab2fcee0f4f46f
Successfully built transformers
  Attempting uninstall: huggingface-hub
    Found 

In [3]:
import torch
import transformers
import peft
import accelerate

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
Transformers: 5.25.0.dev0
PEFT: 0.22.0
Accelerate: 1.16.0
GPU: Tesla T4


In [4]:
import os

for file in os.listdir("/content"):
    print(file)

.config
sample_data


In [5]:
import os

print(os.listdir("/content"))

['.config', 'adaption_qwen3_5_2b_gramasetu_rural_commerce_2892bb55.tgz', 'sample_data']


In [7]:
!pip install zstandard

In [9]:
import os
import zstandard as zstd

path = "/content/adaption_qwen3_5_2b_gramasetu_rural_commerce_2892bb55.tgz"

print("Archive exists:", os.path.exists(path))
print("Archive size:", round(os.path.getsize(path) / (1024**2), 2), "MB")

with open(path, "rb") as f:
    print("File signature:", f.read(16).hex())

print("Zstandard check:")

try:
    with open(path, "rb") as f:
        reader = zstd.ZstdDecompressor().stream_reader(f)
        data = reader.read()
        reader.close()
    print("Decompression successful:", len(data), "bytes")
except Exception as e:
    print("Decompression failed:", repr(e))

Archive exists: True
Archive size: 24.0 MB
File signature: 28b52ffd04582c860a0eb09569502d10
Zstandard check:
Decompression successful: 27131904 bytes


In [11]:
import os
import tarfile

path = "/content/gramasetu_decompressed.bin"

print("Decompressed file size:", os.path.getsize(path), "bytes")

try:
    with tarfile.open(path, mode="r|") as tar:
        for i, member in enumerate(tar):
            print(i + 1, member.name, member.size)
except Exception as e:
    print("Archive reading stopped:", repr(e))

Decompressed file size: 48758784 bytes
1 . 0
2 ./README.md 5101
3 ./adapter_config.json 10254
4 ./adapter_model.safetensors 174616272
Archive reading stopped: ReadError('unexpected end of data')


In [12]:
import os

path = "/content/adaption_qwen3_5_2b_gramasetu_rural_commerce_2892bb55.tgz"

print("File exists:", os.path.exists(path))
if os.path.exists(path):
    print("File size:", round(os.path.getsize(path) / (1024**2), 2), "MB")

File exists: True
File size: 157.67 MB


In [13]:
import os
import zstandard as zstd
import tarfile

source = "/content/adaption_qwen3_5_2b_gramasetu_rural_commerce_2892bb55.tgz"
output = "/content/gramasetu_full.tar"
extract_path = "/content/gramasetu_model"

with open(source, "rb") as compressed:
    with open(output, "wb") as decompressed:
        zstd.ZstdDecompressor().copy_stream(compressed, decompressed)

os.makedirs(extract_path, exist_ok=True)

with tarfile.open(output, "r:") as archive:
    archive.extractall(extract_path, filter="data")

print("Extraction completed!")

for root, dirs, files in os.walk(extract_path):
    for name in files:
        print(name, os.path.getsize(os.path.join(root, name)))

Extraction completed!
adapter_model.safetensors 174616272
trainer_state.json 176362
special_tokens_map.json 325
preprocessor_config.json 443
adapter_config.json 10254
tokenizer.json 19989325
tokenizer_config.json 1192
README.md 5101
chat_template.jinja 7650
processor_config.json 1191
config.json 2734


In [14]:
import json

folder = "/content/gramasetu_model"

for filename in ["adapter_config.json", "config.json"]:
    print("\n", filename)
    with open(f"{folder}/{filename}", "r") as f:
        data = json.load(f)

    if filename == "adapter_config.json":
        print("Base model:", data.get("base_model_name_or_path"))
        print("LoRA rank:", data.get("r"))
        print("LoRA alpha:", data.get("lora_alpha"))
        print("Target modules:", data.get("target_modules"))
    else:
        print("Model architecture:", data.get("model_type"))
        print("Architectures:", data.get("architectures"))


 adapter_config.json
Base model: togethercomputer/Qwen3.5-2B
LoRA rank: 64
LoRA alpha: 128
Target modules: ['gate_proj', 'up_proj', 'o_proj', 'down_proj', 'v_proj', 'q_proj', 'k_proj']

 config.json
Model architecture: qwen3_5
Architectures: ['Qwen3_5ForConditionalGeneration']


In [15]:
from huggingface_hub import HfApi

model_id = "togethercomputer/Qwen3.5-2B"

try:
    info = HfApi().model_info(model_id)
    print("Repository found:", info.id)
    print("Repository status: Accessible")
except Exception as e:
    print("Repository check failed:")
    print(type(e).__name__, str(e))

Repository check failed:
RepositoryNotFoundError 401 Client Error. (Request ID: Root=1-6b096529-7319dd464e07541203b7e17c;a2318fb8-4864-4c84-b9fb-e2f2be31b78a)

Repository Not Found for url: https://huggingface.co/api/models/togethercomputer/Qwen3.5-2B.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.


In [16]:
from huggingface_hub import HfApi

model_id = "Qwen/Qwen3.5-2B"

try:
    info = HfApi().model_info(model_id)
    print("Repository:", info.id)
    print("Status: Accessible")
    print("Architecture:", info.config.get("model_type"))
except Exception as e:
    print(type(e).__name__, str(e))

Repository: Qwen/Qwen3.5-2B
Status: Accessible
Architecture: qwen3_5


In [17]:
import json

path = "/content/gramasetu_model/adapter_config.json"

with open(path, "r") as f:
    config = json.load(f)

print(json.dumps(config, indent=2))

{
  "alpha_pattern": {},
  "auto_mapping": null,
  "base_model_name_or_path": "togethercomputer/Qwen3.5-2B",
  "bias": "none",
  "corda_config": null,
  "eva_config": null,
  "exclude_modules": [
    "model.visual.blocks.5.attn.proj",
    "model.visual.blocks.6.mlp",
    "model.visual.blocks.14.mlp.linear_fc1",
    "model.visual.blocks.15.attn",
    "model.visual.blocks.21.norm1",
    "model.visual.blocks.2.mlp.linear_fc2",
    "model.visual.blocks.14.norm2",
    "model.visual.blocks.20.mlp",
    "model.visual.blocks.14",
    "model.visual.blocks.6.attn",
    "model.visual.blocks.17.mlp.linear_fc1",
    "model.visual.blocks.13.attn.qkv",
    "model.visual.blocks.3.attn.proj",
    "model.visual.blocks",
    "model.visual.blocks.0.mlp.linear_fc1",
    "model.visual.blocks.7.mlp.linear_fc2",
    "model.visual.blocks.9.mlp.act_fn",
    "model.visual.blocks.10.mlp.linear_fc1",
    "model.visual.blocks.21.mlp.linear_fc2",
    "model.visual.blocks.5.mlp.linear_fc2",
    "model.visual.blocks.1

In [18]:
import json

path = "/content/gramasetu_model/adapter_config.json"

with open(path, "r") as f:
    config = json.load(f)

config["base_model_name_or_path"] = "Qwen/Qwen3.5-2B"

with open(path, "w") as f:
    json.dump(config, f, indent=2)

print("Updated base model:", config["base_model_name_or_path"])
print("LoRA rank:", config["r"])
print("LoRA alpha:", config["lora_alpha"])

Updated base model: Qwen/Qwen3.5-2B
LoRA rank: 64
LoRA alpha: 128


In [19]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM
from peft import PeftModel

print("PyTorch:", torch.__version__)
print("Multimodal model class:", AutoModelForMultimodalLM)
print("PEFT:", PeftModel)
print("GPU available:", torch.cuda.is_available())

PyTorch: 2.14.1+cu130
Multimodal model class: <class 'transformers.models.auto.modeling_auto.AutoModelForMultimodalLM'>
PEFT: <class 'peft.peft_model.PeftModel'>
GPU available: True


In [20]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

BASE_MODEL_ID = "Qwen/Qwen3.5-2B"

processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)

base_model = AutoModelForMultimodalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

print("Base model loaded successfully!")
print("Model:", BASE_MODEL_ID)
print("GPU:", torch.cuda.get_device_name(0))
print("Model device:", base_model.device)

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 4.55GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

Base model loaded successfully!
Model: Qwen/Qwen3.5-2B
GPU: Tesla T4
Model device: cuda:0


In [21]:
from peft import PeftModel

ADAPTER_PATH = "/content/gramasetu_model"

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
    is_trainable=False
)

model.eval()

print("GramaSetu AI adapter loaded successfully!")
print("Adapter:", ADAPTER_PATH)
print("Model device:", model.device)

/usr/local/lib/python3.13/dist-packages/peft/import_utils.py:195: UserWarning: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported. torchao support in PEFT is disabled; if you want to use a torchao-quantized model with PEFT, please upgrade torchao.
  warnings.warn(


GramaSetu AI adapter loaded successfully!
Adapter: /content/gramasetu_model
Model device: cuda:0


In [22]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "A group of 20 farmers wants to purchase 500 kg of fertilizer. The normal price is ₹50 per kg, but a bulk supplier offers ₹45 per kg. Calculate the total cost at both prices and the total savings. Explain how GramaSetu AI can help coordinate this group purchase."
            }
        ]
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {
    key: value.to(model.device) if isinstance(value, torch.Tensor) else value
    for key, value in inputs.items()
}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False
    )

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("GRAMASETU AI RESPONSE:\n")
print(response)

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is either not installed or doesn't support the device of the inputs. This is correct but much slower.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is either not installed or doesn't support the device of the inputs. This is correct but much slower.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is either not installed or doesn't support the device of the inputs. This is correct but much slower.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is either not installed or doesn't support the device of the inputs. This is correct but much slower.


GRAMASETU AI RESPONSE:

### Financial Analysis: Bulk Purchase vs. Normal Price

Here is the step-by-step calculation for the total cost and savings based on the provided data.

**1. Calculate Total Cost at Normal Price**
*   **Quantity:** 500 kg
*   **Price per kg:** ₹50
*   **Calculation:** 500 kg × ₹50 = **₹25,000**

**2. Calculate Total Cost at Bulk Price**
*   **Quantity:** 500 kg
*   **Price per kg:** ₹45
*   **Calculation:** 500 kg × ₹45 = **₹22,500**

**3. Calculate Total Savings**
*   **Calculation:** ₹25,000 (Normal) - ₹22,500 (Bulk) = **₹2,500**
*   **Savings per kg:** ₹50 - ₹45 = ₹5 per kg
*   **Total Savings:** 500 kg × ₹5 = **₹2,500**

**Summary:** By coordinating a group purchase, the farmers save **₹2,500** (a 10% reduction in expenditure).

---

### How GramaSetu AI Can Help Coordinate This Group Purchase

GramaSetu AI is designed to bridge


In [23]:
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=600,
        do_sample=False
    )

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

### Financial Analysis: Bulk Purchase vs. Normal Price

Here is the step-by-step calculation for the total cost and savings based on the provided data.

**1. Calculate Total Cost at Normal Price**
*   **Quantity:** 500 kg
*   **Price per kg:** ₹50
*   **Calculation:** 500 kg × ₹50 = **₹25,000**

**2. Calculate Total Cost at Bulk Price**
*   **Quantity:** 500 kg
*   **Price per kg:** ₹45
*   **Calculation:** 500 kg × ₹45 = **₹22,500**

**3. Calculate Total Savings**
*   **Calculation:** ₹25,000 (Normal) - ₹22,500 (Bulk) = **₹2,500**
*   **Savings per kg:** ₹50 - ₹45 = ₹5 per kg
*   **Total Savings:** 500 kg × ₹5 = **₹2,500**

**Summary:** By coordinating a group purchase, the farmers save **₹2,500** (a 10% reduction in expenditure).

---

### How GramaSetu AI Can Help Coordinate This Group Purchase

GramaSetu AI is designed to bridge the gap between individual smallholder farmers and large-scale supply chains. In this scenario, it can assist the group of 20 farmers in the following ways

In [24]:
test_prompt = """
A village Kirana shop sells 40 packets of rice per day.
Currently, it has 120 packets in stock.
Its supplier takes 4 days to deliver new stock.
The shopkeeper wants to maintain 3 extra days of safety stock.

Calculate:
1. How many days will the current stock last?
2. What is the reorder point?
3. When should the shopkeeper place the next order?

Explain how GramaSetu AI can help prevent stockouts and reduce excess inventory.
"""

messages = [
    {
        "role": "user",
        "content": [{"type": "text", "text": test_prompt}]
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {
    k: v.to(model.device) if isinstance(v, torch.Tensor) else v
    for k, v in inputs.items()
}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=False
    )

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

### 1. Calculation of Days Until Stockout
To determine how long the current stock will last, divide the total inventory by the daily consumption rate.
*   **Current Stock:** 120 packets
*   **Daily Consumption:** 40 packets
*   **Calculation:** 120 ÷ 40 = 3 days

**Answer:** The current stock will last for **3 days**.

---

### 2. Calculation of Reorder Point (ROP)
The Reorder Point is the specific inventory level at which a new order must be placed to avoid stockouts. It is calculated by adding the safety stock to the current stock.
*   **Current Stock:** 120 packets
*   **Safety Stock:** 3 packets (3 extra days × 40 packets/day)
*   **Calculation:** 120 + 3 = 123 packets

**Answer:** The reorder point is **123 packets**.

---

### 3. Timing for the Next Order
The shopkeeper should place the next order immediately when the inventory level drops to or below the reorder point.
*   **Trigger:** Once the stock reaches 123 packets.
*   **Action:** Place an order for 123 packets (assuming t

In [25]:
test_prompt = """
A rural farmers group needs to transport 1,200 kg of vegetables to a nearby market.

Transport options:
Option A: Small vehicle, capacity 400 kg, cost ₹1,500 per trip.
Option B: Large vehicle, capacity 1,200 kg, cost ₹3,800 per trip.

Calculate:
1. Number of trips required for each option.
2. Total transportation cost for each option.
3. Which option is cheaper?
4. Total savings using the cheaper option.

Explain how GramaSetu AI can coordinate transportation for the farmers and reduce logistics costs.
Show all calculations clearly.
"""

messages = [
    {
        "role": "user",
        "content": [{"type": "text", "text": test_prompt}]
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {
    k: v.to(model.device) if isinstance(v, torch.Tensor) else v
    for k, v in inputs.items()
}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=False
    )

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

### 1. Number of Trips Required

To determine the number of trips, we divide the total weight of the vegetables by the capacity of each vehicle option.

*   **Option A (Small Vehicle):**
    *   Total Weight: 1,200 kg
    *   Capacity: 400 kg
    *   Calculation: $1,200 \text{ kg} \div 400 \text{ kg} = 3$ trips
*   **Option B (Large Vehicle):**
    *   Total Weight: 1,200 kg
    *   Capacity: 1,200 kg
    *   Calculation: $1,200 \text{ kg} \div 1,200 \text{ kg} = 1$ trip

### 2. Total Transportation Cost

Next, we calculate the total cost for each option by multiplying the number of trips by the cost per trip.

*   **Option A (Small Vehicle):**
    *   Trips: 3
    *   Cost per Trip: ₹1,500
    *   Calculation: $3 \text{ trips} \times ₹1,500 = ₹4,500$
*   **Option B (Large Vehicle):**
    *   Trips: 1
    *   Cost per Trip: ₹3,800
    *   Calculation: $1 \text{ trip} \times ₹3,800 = ₹3,800$

### 3. Which Option is Cheaper?

Comparing the total costs:
*   Option A: ₹4,500
*   Option B: 

In [26]:
test_prompt = """
A farmer harvests 800 kg of tomatoes.

Selling options:
Option A: Sell to a local trader at ₹18 per kg with no transportation cost.
Option B: Sell directly in a town market at ₹25 per kg, but transportation costs ₹2,000 and market fees are ₹500.

Calculate:
1. Total revenue under Option A.
2. Total revenue under Option B before expenses.
3. Net earnings under both options after expenses.
4. Which option is more profitable?
5. How much additional profit does the farmer earn using the better option?

Explain how GramaSetu AI can help farmers access markets, compare buyers, and reduce dependence on intermediaries.
"""

messages = [
    {
        "role": "user",
        "content": [{"type": "text", "text": test_prompt}]
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {
    k: v.to(model.device) if isinstance(v, torch.Tensor) else v
    for k, v in inputs.items()
}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=600,
        do_sample=False
    )

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

### 1. Total Revenue Calculation

**Option A: Local Trader**
*   **Formula:** Weight × Selling Price
*   **Calculation:** 800 kg × ₹18/kg
*   **Total Revenue:** **₹14,400**

**Option B: Town Market**
*   **Formula:** Weight × Selling Price
*   **Calculation:** 800 kg × ₹25/kg
*   **Total Revenue:** **₹20,000**

---

### 2. Net Earnings Calculation

To find the net earnings, we subtract the specified expenses from the total revenue.

**Option A: Local Trader**
*   **Total Revenue:** ₹14,400
*   **Transportation Cost:** ₹0
*   **Market Fees:** ₹0
*   **Net Earnings:** ₹14,400 - ₹0 - ₹0 = **₹14,400**

**Option B: Town Market**
*   **Total Revenue:** ₹20,000
*   **Transportation Cost:** ₹2,000
*   **Market Fees:** ₹500
*   **Total Expenses:** ₹2,000 + ₹500 = ₹2,500
*   **Net Earnings:** ₹20,000 - ₹2,500 = **₹17,500**

---

### 3. Profitability Comparison

**Which option is more profitable?**
**Option B (Town Market)** is more profitable. While Option A offers a higher per-kg price, the sig

In [27]:
test_prompt = """
A rural artisan group produces 150 handmade baskets.

They have two selling options:

Option A: Sell to a local wholesaler at ₹120 per basket with no platform commission.
Option B: Sell directly through an online marketplace at ₹180 per basket. The marketplace charges a 10% commission on the selling price per basket. Packaging and shipping cost ₹1,500 in total.

Calculate:
1. Total revenue under Option A.
2. Gross revenue under Option B.
3. Total marketplace commission.
4. Net earnings under both options after applicable expenses.
5. Which option is more profitable?
6. Additional earnings from choosing the better option.

Explain how GramaSetu AI can help rural artisans reach customers, price their products, and manage online orders.
Show all calculations clearly.
"""

messages = [
    {
        "role": "user",
        "content": [{"type": "text", "text": test_prompt}]
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {
    k: v.to(model.device) if isinstance(v, torch.Tensor) else v
    for k, v in inputs.items()
}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=600,
        do_sample=False
    )

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

### 1. Total Revenue under Option A
To determine the total revenue under Option A, we multiply the number of baskets produced by the wholesale price per basket.
*   **Formula:** Revenue = Quantity × Selling Price
*   **Calculation:** 150 baskets × ₹120 per basket
*   **Total Revenue:** **₹18,000**

### 2. Gross Revenue under Option B
Gross revenue represents the total amount received from the customer before deducting marketplace fees.
*   **Formula:** Gross Revenue = Quantity × Selling Price
*   **Calculation:** 150 baskets × ₹180 per basket
*   **Total Gross Revenue:** **₹27,000**

### 3. Total Marketplace Commission
The marketplace charges a 10% commission on the selling price per basket.
*   **Formula:** Commission = Gross Revenue × Commission Rate
*   **Calculation:** ₹27,000 × 10%
*   **Total Commission:** **₹2,700**

### 4. Net Earnings under Both Options
Net earnings are calculated by subtracting total expenses from the gross revenue.
*   **Common Expenses:** Packaging and ship

In [28]:
test_prompt = """
A Women's Self-Help Group (SHG) has 12 members who produce handmade cloth bags.

Each member produces 25 bags per month.
The selling price is ₹80 per bag.
The total monthly expenses for raw materials, packaging, and transportation are ₹8,000.

Calculate:
1. Total bags produced per month.
2. Total monthly sales revenue.
3. Net monthly profit after expenses.
4. Equal profit share per member.
5. Monthly income per member if each member also receives an equal share of the profit.

Assume all bags are sold and expenses are shared equally. Clearly distinguish sales revenue, profit, and total income per member.

Explain how GramaSetu AI can help SHGs manage production, orders, payments, and market access.
"""

messages = [
    {
        "role": "user",
        "content": [{"type": "text", "text": test_prompt}]
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {
    k: v.to(model.device) if isinstance(v, torch.Tensor) else v
    for k, v in inputs.items()
}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=600,
        do_sample=False
    )

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

### 1. Financial Calculations for the Women's SHG

Here is the step-by-step breakdown of the financial performance based on the provided data:

*   **Total bags produced per month:** 12 members × 25 bags/member = **300 bags**
*   **Total monthly sales revenue:** 300 bags × ₹80/bag = **₹24,000**
*   **Net monthly profit after expenses:** ₹24,000 (Revenue) - ₹8,000 (Expenses) = **₹16,000**
*   **Equal profit share per member:** ₹16,000 ÷ 12 members = **₹1,333.33**
*   **Monthly income per member (Profit share):** **₹1,333.33**

*(Note: The total monthly income per member includes both the profit share and the equal share of the fixed expenses. If calculating only the profit share, it is ₹1,333.33. If calculating total income per member, it is ₹1,333.33 + ₹666.67 = ₹2,000.)*

---

### 2. Detailed Explanations for GramaSetu AI

GramaSetu AI is a digital platform designed to bridge the gap between rural artisans and urban markets. For a Self-Help Group (SHG) like this, the platform acts as 

In [29]:
test_prompt = """
A rural employment initiative trains 40 unemployed young people.

After training:
- 30 successfully complete the program.
- 24 of those who complete training get jobs.
- Each employed person earns ₹12,000 per month.

Calculate:
1. Training completion rate as a percentage of enrolled participants.
2. Employment rate among those who completed training.
3. Overall employment rate among all enrolled participants.
4. Total monthly income generated by the employed youth.
5. Number of trained participants who remain unemployed.

Explain how GramaSetu AI can connect rural youth with skill training, local businesses, job opportunities, and employers.

Show all calculations and distinguish the different employment rates.
"""

messages = [
    {
        "role": "user",
        "content": [{"type": "text", "text": test_prompt}]
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {
    k: v.to(model.device) if isinstance(v, torch.Tensor) else v
    for k, v in inputs.items()
}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=600,
        do_sample=False
    )

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

### 1. Calculations and Explanations

Here is the step-by-step breakdown of the employment metrics for the initiative.

**Given Data:**
*   **Total Enrolled Participants:** 40
*   **Successfully Completed Training:** 30
*   **Jobs Obtained by Completed Participants:** 24
*   **Monthly Income per Employed Youth:** ₹12,000

**Step 1: Training Completion Rate**
This measures the success of the training curriculum itself.
*   **Formula:** (Number of Completed Participants / Total Enrolled Participants) × 100
*   **Calculation:** (30 / 40) × 100 = **75%**

**Step 2: Employment Rate Among Completed Participants**
This measures the effectiveness of the placement services for those who finished the training.
*   **Formula:** (Number of Jobs Obtained by Completed / Number of Completed Participants) × 100
*   **Calculation:** (24 / 30) × 100 = **80%**

**Step 3: Overall Employment Rate Among Enrolled Participants**
This measures the total success rate of the initiative from the start to finish.
